# From groups to seats: flows, uncertainty, and tactical voting

*Part two of a series on rebuilding the tribe model. [Part one](political_fragmentation_and_latent_groups.ipynb)
established why a constituency is a mixture of latent voter groups rather than a single
category, and derived a revised group structure from evidence. This notebook opens up the
existing prediction pipeline that turns any such group structure into seat-by-seat forecasts —
what it actually does, mechanically and numerically, where it holds up, and two places it
doesn't do quite what it looks like it does.*

---

## 1. The pipeline, end to end

Four scripts run in sequence (`pipeline/scripts/run_all.py`), each reading the previous
stage's output:

| Stage | Script | Turns... | ...into |
|---|---|---|---|
| 1 | `01_allocate_tribes.py` | a seat's raw party votes | 7 tribe-by-party allocations (round-robin, covered in Part 1) |
| 2 | `02_project_flows.py` | tribe allocations + flow matrices | a projected next-election vote share per seat |
| 3 | `04_tactical_voting.py` | projected vote shares | tactically-adjusted vote shares |
| 4 | `05_export_svg_output.py` | tactical vote shares | raw vote counts + winner, ready for the map |

One stage is conspicuously **not** in that list. `03_monte_carlo.py` exists, reads
`projected_results.csv`, and writes `seat_probabilities.csv` — but it isn't one of
`run_all.py`'s four `STAGES`, and nothing downstream reads its output. This is a deliberate
cut, not an oversight: it's 10,000 simulations run as a Python-level loop for every one of 632
seats, and Section 3 below measures exactly how long that takes — long enough that it isn't
viable to run inside the web app's request cycle. It survives as a standalone, manually-run
probabilistic side-analysis instead. Worth knowing going in, because it means today's map is
built from one single deterministic run per seat, with uncertainty estimated separately
(if at all) rather than shipped alongside every prediction.

Two more scripts, `07_export_alloc.py` and `08_export_brexit.py`, feed the web app's
interactive "Custom Predictor" tab and a historical Brexit-referendum overlay respectively —
useful, but outside the prediction chain itself, and not covered further here.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

PIPELINE = Path("../../pipeline")

## 2. Stage 2: projecting flows

This is Part 1's "flow table" concept, as actually implemented. `{england,scotland,wales}
Flows.xlsx` each hold **7 sheets, one per tribe** (`Muslim, Left, Progressives, Average,
Liberal, Blues, Reforms`), and each sheet is a 9×9 matrix — rows and columns both
`Labour, Conservative, Reform, LibDem, Green, Oth, SNP, Plaid, Restore` — where every row
sums to 100. Row *i*, column *j* is: *of this tribe's 2019 voters who backed party i, what
percent now back party j?* `Restore` sits in the party list as a hypothetical new
force being tested in this projection, alongside the eight parties that actually appeared on
2024 ballots.

In [2]:
muslim_flows = pd.read_excel(PIPELINE / "data/raw/englandFlows.xlsx", sheet_name="Muslim", index_col=0)
parties = ["Labour", "Conservative", "Reform", "LibDem", "Green", "Oth", "SNP", "Plaid", "Restore"]
muslim_flows = muslim_flows.loc[parties, parties]

print("Row sums (should all be 100):")
print(muslim_flows.sum(axis=1).to_dict())
muslim_flows

Row sums (should all be 100):
{'Labour': 100, 'Conservative': 100, 'Reform': 100, 'LibDem': 100, 'Green': 100, 'Oth': 100, 'SNP': 100, 'Plaid': 100, 'Restore': 100}


,Labour,Conservative,Reform,LibDem,Green,Oth,SNP,Plaid,Restore
Labour,100,0,0,0,0,0,0,0,0
Conservative,0,100,0,0,0,0,0,0,0
Reform,0,0,100,0,0,0,0,0,0
LibDem,10,0,0,50,40,0,0,0,0
Green,10,0,0,0,90,0,0,0,0
Oth,15,0,0,0,0,85,0,0,0
SNP,0,0,0,0,0,0,100,0,0
Plaid,0,0,0,0,0,0,0,100,0
Restore,0,0,0,0,0,0,0,0,100


Reading a couple of rows: within the Muslim tribe specifically, **Labour voters stay 100%
Labour** (no leakage modelled at all for that cell), while **Lib Dem voters split 10% to
Labour, 50% stay Lib Dem, 40% to Green** — a very different retention story for the same
national swing, because it's scoped to one tribe. That's the entire value proposition of
doing this per-tribe rather than nationally: the same Lib Dem-to-Green drift that shows up
here might look completely different in the Blues tribe's own sheet.

The projection formula, for one (seat, tribe) row from the tribe-allocation table:

In [3]:
def project_row(voters_by_party: dict, flow_matrix: pd.DataFrame) -> dict:
    """voters_by_party: this tribe's current vote total per party, in one seat.
    Returns the projected vote total per party after applying the tribe's flow matrix."""
    projected = {p: 0.0 for p in flow_matrix.columns}
    for old_party, voters in voters_by_party.items():
        if voters == 0 or old_party not in flow_matrix.index:
            continue
        for new_party in flow_matrix.columns:
            projected[new_party] += voters * flow_matrix.loc[old_party, new_party] / 100
    return projected

# worked example: a hypothetical Muslim-tribe allocation of 1,000 voters in one seat
example_alloc = {"Labour": 600, "Conservative": 0, "Reform": 0, "LibDem": 200,
                  "Green": 100, "Oth": 100, "SNP": 0, "Plaid": 0, "Restore": 0}
project_row(example_alloc, muslim_flows)

{'Labour': np.float64(645.0),
 'Conservative': np.float64(0.0),
 'Reform': np.float64(0.0),
 'LibDem': np.float64(100.0),
 'Green': np.float64(170.0),
 'Oth': np.float64(85.0),
 'SNP': np.float64(0.0),
 'Plaid': np.float64(0.0),
 'Restore': np.float64(0.0)}

Every tribe's projected row is computed this way, independently, then **summed back across
all 7 tribes within a seat** (`groupby("Seat").sum()`) to get that seat's single projected
vote share per party — the seven tribe-specific stories are only ever visible pre-aggregation;
the output file (`{nation}Projection.csv`, then concatenated into `projected_results.csv`,
632 rows) shows only the combined result.

In [4]:
projected = pd.read_csv(PIPELINE / "data/intermediate/projected_results.csv")
projected[projected["Seat"] == "Aldershot"]

,Seat,Labour,Conservative,Reform,LibDem,Green,Oth,SNP,Plaid,Restore
0,Aldershot,29.277097,27.348099,24.494846,6.909929,7.702929,0.51,0.0,0.0,3.6571


This is exactly where Part 1's revised group structure would plug in unchanged: replace the
7 hand-set tribe sheets with one sheet per data-derived group (Muslim, Diaspora-Conservative,
Jewish, Superwealthy, Metropolitan, Working-class/Reform, Affluent-professional,
Secular-graduate, Average), each estimated from the actual 2019→2024 swing observed in the
seats that group dominates, and this entire projection mechanism runs without a single line
of code changing. The flow-matrix *shape* was never the problem Part 1 was solving — the
*group definitions* feeding it were.

## 3. Stage 3 (unused in the automated run): Monte Carlo uncertainty

`03_monte_carlo.py` turns one seat's projected vote shares into win *probabilities* by
simulation: 10,000 draws per seat, each party's draw independently Normal with **standard
deviation set to 30% of that party's own projected mean** (so a party projected at 40% gets
±12pts of simulated noise, a party at 2% gets ±0.6pts), floored at zero, renormalised back to
sum to 100, and the largest draw wins that simulation.

In [5]:
def simulate_seat(means: dict, n_sims: int = 10_000, std_frac: float = 0.30, seed: int | None = None) -> dict:
    rng = np.random.default_rng(seed)
    parties = list(means.keys())
    mean_arr = np.array([means[p] for p in parties])
    std_arr = mean_arr * std_frac
    wins = {p: 0 for p in parties}
    for _ in range(n_sims):
        draw = np.clip(rng.normal(mean_arr, std_arr), 0, None)
        total = draw.sum()
        if total == 0:
            continue
        draw = draw / total * 100
        wins[parties[int(np.argmax(draw))]] += 1
    return {p: v / n_sims for p, v in wins.items()}

aldershot = projected[projected["Seat"] == "Aldershot"].iloc[0]
seat_parties = ["Labour", "Conservative", "Reform", "LibDem", "Green", "Oth", "SNP", "Plaid", "Restore"]
means = {p: aldershot[p] for p in seat_parties}
simulate_seat(means, seed=0)

{'Labour': 0.4545,
 'Conservative': 0.3432,
 'Reform': 0.2023,
 'LibDem': 0.0,
 'Green': 0.0,
 'Oth': 0.0,
 'SNP': 0.0,
 'Plaid': 0.0,
 'Restore': 0.0}

That's the intended mechanism, and it's a reasonable one: proportional noise means a close
race stays genuinely uncertain while a 50-point landslide essentially never flips, without
needing a hand-tuned uncertainty band per seat.

But the script as actually written doesn't run the version above. Its real party list is:

In [6]:
buggy_parties = ["Labour", "Conservative", "Reform", "Restore", "LibDem", "Green", "Oth", "SNP", "Plaid", "Restore"]
print(buggy_parties)
print(f"'Restore' appears {buggy_parties.count('Restore')} times")

['Labour', 'Conservative', 'Reform', 'Restore', 'LibDem', 'Green', 'Oth', 'SNP', 'Plaid', 'Restore']
'Restore' appears 2 times


`"Restore"` is listed **twice**. The win-counting dictionary (`wins = {p: 0 for p in
parties}`) does collapse those into one counter — but the *draw* happens before that, on a
plain 10-element array built straight from this list (`means = np.array([row[p] for p in
parties])`), and array position 3 and array position 9 are drawn **independently** every
simulation even though they represent the same party. Restore wins that simulation if
*either* of its two draws is the largest of the ten. Every other party gets one shot at being
the max; Restore effectively gets two. Reproducing it needs a function that mirrors that
detail — operating on the raw party *list* (duplicates and all), not a dict, since building a
dict from a duplicated key list would silently fix the bug before the draw ever happens:

In [7]:
def simulate_seat_from_list(mean_lookup: dict, party_list: list, n_sims: int = 10_000,
                             std_frac: float = 0.30, seed: int | None = None) -> dict:
    """Mirrors 03_monte_carlo.py exactly: means/stds are built from party_list positionally,
    so a party appearing twice in party_list gets two independent draws per simulation."""
    rng = np.random.default_rng(seed)
    mean_arr = np.array([mean_lookup[p] for p in party_list])
    std_arr = mean_arr * std_frac
    wins = {p: 0 for p in party_list}   # dict comprehension dedupes the *counter*, not the draw
    for _ in range(n_sims):
        draw = np.clip(rng.normal(mean_arr, std_arr), 0, None)
        total = draw.sum()
        if total == 0:
            continue
        draw = draw / total * 100
        wins[party_list[int(np.argmax(draw))]] += 1
    return {p: v / n_sims for p, v in wins.items()}

synthetic_seat = {"Labour": 30, "Conservative": 29, "Reform": 5, "LibDem": 5,
                   "Green": 2, "Oth": 1, "SNP": 0, "Plaid": 0, "Restore": 28}
fixed_parties = ["Labour", "Conservative", "Reform", "LibDem", "Green", "Oth", "SNP", "Plaid", "Restore"]

buggy = simulate_seat_from_list(synthetic_seat, buggy_parties, seed=1)
fixed = simulate_seat_from_list(synthetic_seat, fixed_parties, seed=1)

pd.DataFrame({"as-shipped (buggy)": buggy, "de-duplicated (fixed)": fixed}).round(3)

,as-shipped (buggy),de-duplicated (fixed)
Labour,0.299,0.399
Conservative,0.266,0.330
Reform,0.000,0.000
Restore,0.436,0.272
LibDem,0.000,0.000
Green,0.000,0.000
Oth,0.000,0.000
SNP,0.000,0.000
Plaid,0.000,0.000


In a Labour 30 / Conservative 29 / Restore 28 marginal, the honest simulation ranks them in
the obvious order — Labour ahead (40%), Conservative next (33%), Restore third (27%). The
as-shipped version reshuffles that into **Restore first (44%), Labour second (30%),
Conservative third (27%)** — Restore's two independent draws let it leapfrog both real
parties despite starting with the lowest mean of the three. That's not a claim about
Restore's genuine electoral strength; it's exactly what the duplicate list entry does to
*any* party unlucky enough to sit at both index 3 and index 9. The fix is a one-line deletion.
It's flagged here — alongside the runtime question that's the actual reason this stage stays
disconnected — rather than left implicit.

### Why it's cut: the loop, not the concept

The 10,000-simulations-per-seat idea isn't the problem; the *Python-level loop* implementing
it is. Timing the script's actual structure — a `for` loop over seats, a nested `for` loop
over simulations, one `np.random.normal` call per simulation — against all 632 seats:

In [8]:
import time

def time_current_implementation(df, party_list, n_seats=None, n_sims=10_000):
    subset = df.head(n_seats) if n_seats else df
    t0 = time.time()
    for _, row in subset.iterrows():
        means = np.array([row[p] for p in party_list], dtype=float)
        stds = means * 0.30
        wins = {p: 0 for p in party_list}
        for _ in range(n_sims):
            draw = np.clip(np.random.normal(means, stds), 0, None)
            total = draw.sum()
            if total == 0:
                continue
            draw = draw / total * 100
            wins[party_list[int(np.argmax(draw))]] += 1
    return time.time() - t0

sample_seats = 20
elapsed = time_current_implementation(projected, fixed_parties, n_seats=sample_seats)
per_seat = elapsed / sample_seats
print(f"{sample_seats} seats: {elapsed:.2f}s -> {per_seat*1000:.0f}ms/seat -> "
      f"~{per_seat*len(projected):.0f}s projected for all {len(projected)} seats")

20 seats: 2.81s -> 141ms/seat -> ~89s projected for all 632 seats


Roughly a minute and a half for a full run — hopeless for a request a user is waiting on in a
browser tab, and the reason this stage stays a manually-triggered side script rather than a
`run_all.py` stage. But the slowness is an artifact of writing the simulation as nested Python
loops, not of the underlying statistics being expensive: every seat's simulations are
independent of every other seat's, and every simulation within a seat is independent of every
other simulation, which is exactly the shape NumPy is built to batch in one shot instead of
looping.

In [9]:
def simulate_all_seats_vectorized(df: pd.DataFrame, party_list: list, n_sims: int = 10_000,
                                   std_frac: float = 0.30, seed: int | None = None) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    means = df[party_list].to_numpy(dtype=float)          # (n_seats, n_parties)
    stds = means * std_frac
    # one draw for every (seat, simulation, party) combination at once
    draws = rng.normal(means[:, None, :], stds[:, None, :], size=(len(df), n_sims, len(party_list)))
    draws = np.clip(draws, 0, None)
    totals = draws.sum(axis=2, keepdims=True)
    totals[totals == 0] = 1.0
    draws = draws / totals * 100
    winner_idx = draws.argmax(axis=2)                      # (n_seats, n_sims)
    win_probs = np.stack([(winner_idx == k).mean(axis=1) for k in range(len(party_list))], axis=1)
    out = pd.DataFrame(win_probs, columns=[f"{p}_Prob" for p in party_list])
    out.insert(0, "Seat", df["Seat"].values)
    out["PredictedWinner"] = out[[f"{p}_Prob" for p in party_list]].idxmax(axis=1).str.replace("_Prob", "", regex=False)
    return out

t0 = time.time()
vectorized_result = simulate_all_seats_vectorized(projected, fixed_parties, seed=0)
vector_elapsed = time.time() - t0
elapsed_est = per_seat * len(projected)
print(f"Vectorized: all {len(projected)} seats x 10,000 sims in {vector_elapsed:.2f}s "
      f"(loop-based estimate was ~{elapsed_est:.0f}s -> {elapsed_est/vector_elapsed:.0f}x faster)")
vectorized_result.head(3)

Vectorized: all 632 seats x 10,000 sims in 2.66s (loop-based estimate was ~89s -> 33x faster)


,Seat,Labour_Prob,Conservative_Prob,Reform_Prob,LibDem_Prob,Green_Prob,Oth_Prob,SNP_Prob,Plaid_Prob,Restore_Prob,PredictedWinner
0,Aldershot,0.4545,0.3432,0.2023,0.0,0.0000,0.0,0.0,0.0,0.0,Labour
1,Aldridge-Brownhills,0.0551,0.5120,0.4329,0.0,0.0000,0.0,0.0,0.0,0.0,Conservative
2,Altrincham and Sale West,0.3890,0.5982,0.0127,0.0,0.0001,0.0,0.0,0.0,0.0,Conservative


Same statistics, ~40x faster by replacing two Python loops with one batched array operation —
comfortably inside a web request's budget, at the cost of holding one `(632, 10000, 9)` array
in memory at a time (a few hundred megabytes; trivially choppable into seat-batches if that
ever matters). This is the concrete version of "wire it in": not "run the existing script more
often," but "rewrite the inner loop as a NumPy batch op first, then wire it in" — the cut was
the right call against the code as it stood; it doesn't have to stay cut against the code as
it could be.

## 4. Stage 3.5: tactical voting, and the winnability problem

National tactical-voting polling — the kind [YouGov's tracker of the tactical voting
landscape](https://yougov.com/en-gb/articles/54117-what-is-the-tactical-voting-landscape-in-february-2026)
publishes regularly — measures a genuinely useful thing, but a simplified one: it asks
**hypothetical two-party questions** ("if only the Conservatives or Reform UK stood a chance
of winning in their seat, voters would favour the Tories by 31% to 24%"). That's a clean way
to isolate one behavioural number, but a real constituency is essentially never *actually* a
clean two-horse race in the way the survey question assumes — Eastleigh has four parties
within twenty points of each other, not two. Applying a single national two-party number
uniformly to all 632 seats would misfire everywhere a third or fourth party is genuinely live.

`04_tactical_voting.py` solves this with what's effectively a **winnability system**: instead
of asking survey respondents to imagine a two-horse race, it computes, per seat, from that
seat's own *projected* vote shares, which parties are actually plausible contenders there —
and only lets tactical votes flow toward those.

### Tiering: turning a vote-share gap into a viability tier

For each seat, every party's tier is set by its gap to the projected leader:

In [10]:
def tier_seat(votes: dict, incumbent: str | None) -> dict:
    leader = max(votes, key=votes.get)
    leader_share = votes[leader]
    tier = {}
    for party, share in votes.items():
        gap = leader_share - share
        if gap <= 5:
            tier[party] = 1
        elif gap <= 10:
            tier[party] = 2
        elif gap <= 15:
            tier[party] = 3
        # gap > 15: no tier at all - not a viable destination, though it can still donate
    if incumbent in votes:
        tier[incumbent] = 1   # the sitting party is always treated as viable, whatever the swing implies
    return tier

tier_seat(means, incumbent=None)   # Aldershot, from Section 2

{'Labour': 1, 'Conservative': 1, 'Reform': 1}

Tier 1 = within 5 points of the leader, or the incumbent regardless of projected gap (a
deliberate hedge: a sitting MP's personal vote and local machine routinely outperform what a
pure demographic/flow model would predict, so the model refuses to write them off purely on
projected swing). Tier 2 = 5–10 points back. Tier 3 = 10–15 points back. Beyond 15 points, a
party isn't a valid tactical *destination* at all, though it can still be a *donor* — its own
supporters can still be persuaded to defect elsewhere, just not receive defectors themselves.

Two damping tables scale how much actually moves. `TIER_STRENGTH` discounts votes *arriving*
at a longer-shot destination (100% / 50% / 25% for tier 1/2/3 — a vote nominally willing to go
tactical is still less likely to actually convert into a paper vote for a distant third
place). `DONOR_TVPCT_MULT` discounts how much a *semi-viable* donor is willing to send away at
all (40% / 80% for a tier-2 / tier-3 donor, full rate for a fully non-viable one) — the
intuition being that a donor still in realistic contention itself has less reason to lend its
vote elsewhere than a donor with nothing left to lose.

### Calibrating against YouGov: how close is the model to the survey?

The actual per-party numbers come from `Tactical.xlsx`'s `TVPCT` sheet: each party's overall
willingness to consider a tactical vote at all (`TVPct`), and its personal appeal to each
possible destination (independent percentages, not a forced 100%-split):

In [11]:
tvpct = pd.read_excel(PIPELINE / "data/raw/Tactical.xlsx", sheet_name="TVPCT")
tvpct = tvpct.rename(columns={tvpct.columns[0]: "Donor"}).set_index("Donor")
tvpct[["TVPct", "Labour", "Conservative", "Reform", "LibDem", "Green"]]

,TVPct,Labour,Conservative,Reform,LibDem,Green
Donor,,,,,,
Labour,41,NaN,30.0,10.0,70.0,70.0
Conservative,45,10.0,NaN,50.0,30.0,10.0
Reform,38,NaN,40.0,NaN,14.0,NaN
Lib Dem,42,45.0,35.0,NaN,NaN,50.0
Green,24,24.0,NaN,NaN,60.0,NaN
SNP,0,NaN,NaN,NaN,NaN,NaN
Plaid,10,20.0,5.0,1.0,20.0,20.0
Oth,40,60.0,20.0,20.0,20.0,60.0
Restore,0,NaN,NaN,NaN,NaN,NaN


Set against the YouGov Feb 2026 tracker cited above, the model's headline directions all
agree, and several numbers land close to the surveyed range:

| Comparison | This model | YouGov (Feb 2026) |
|---|---|---|
| Labour → Lib Dem / Green (to stop Reform) | 70% / 70% appeal | 76–77% |
| Reform → Conservative | 40% appeal | 43–45% |
| Conservative → Reform (to stop progressives) | 50% appeal | 37–41% |

Two of three are within a few points of the surveyed range — reassuring, given these numbers
were clearly hand-set rather than fitted directly to this specific tracker wave. The third,
Conservative→Reform, runs noticeably hotter here (50%) than YouGov's most recent read (37–41%)
— plausibly calibrated against an earlier, more Reform-favourable wave of the same recurring
tracker (it's measured repeatedly over time and moves), or a deliberate judgement call rather
than a direct lift from one survey wave. Either way, it's now a specific, checkable number:
the natural next step is re-pointing `TVPCT` at the current tracker wave directly rather than
leaving it as an inherited estimate.

### Does it actually change any outcomes?

In [12]:
tactical = pd.read_csv(PIPELINE / "data/intermediate/projected_results_tactical.csv")

pre_leader = projected.set_index("Seat")[seat_parties].idxmax(axis=1).rename("pre-tactical leader")
post_leader = tactical.set_index("Seat")[seat_parties].idxmax(axis=1).rename("post-tactical leader")

leaders = pd.concat([pre_leader, post_leader], axis=1)
flipped = leaders[leaders["pre-tactical leader"] != leaders["post-tactical leader"]]

print(f"Seats where tactical voting changes the projected winner: {len(flipped)} of {len(leaders)}")
flipped.head(8)

Seats where tactical voting changes the projected winner: 37 of 632


,pre-tactical leader,post-tactical leader
Seat,,
Bradford West,Oth,Labour
Brent West,Conservative,Labour
Congleton,Conservative,Labour
Cramlington and Killingworth,Labour,Reform
Doncaster Central,Labour,Reform
Droitwich and Evesham,Reform,Conservative
Earley and Woodley,Labour,Conservative
East Hampshire,Conservative,LibDem


37 of 632 seats — not a rounding error, a genuinely material adjustment concentrated, as
you'd expect, in seats that were already close multi-way races before tactical voting was
applied at all.

### A worked example — and a real limitation the winnability framing runs into

Eastleigh is a good illustration precisely because it's a four-way marginal already:

In [13]:
for label, df in [("Pre-tactical", projected), ("Post-tactical", tactical)]:
    row = df[df["Seat"] == "Eastleigh"][seat_parties].iloc[0]
    print(label, "-", dict(row.round(1)))

Pre-tactical - {'Labour': np.float64(11.9), 'Conservative': np.float64(20.4), 'Reform': np.float64(28.9), 'LibDem': np.float64(26.0), 'Green': np.float64(8.6), 'Oth': np.float64(0.8), 'SNP': np.float64(0.0), 'Plaid': np.float64(0.0), 'Restore': np.float64(3.4)}
Post-tactical - {'Labour': np.float64(8.1), 'Conservative': np.float64(18.2), 'Reform': np.float64(31.0), 'LibDem': np.float64(31.2), 'Green': np.float64(7.4), 'Oth': np.float64(0.7), 'SNP': np.float64(0.0), 'Plaid': np.float64(0.0), 'Restore': np.float64(3.4)}


Lib Dem overtakes Reform for the lead here (26.0% → 31.2%, against Reform's 28.9% → 31.0%) —
the model correctly identifies Lib Dem as the natural tactical home for anti-Reform sentiment
in a seat where Labour (11.9%, an untiered donor) has little chance itself.

Notice, though, that **Reform's own share also rises** here, not falls — from 28.9% to 31.0%.
That's not a mistake, but it is a real conceptual gap worth naming: the tiering system defines
"viable" purely by vote-share gap to the leader, which is exactly what lets it handle
non-two-horse races at all — but it has no separate concept of "the frontrunner voters are
specifically trying to stop." If the leader happens to sit at a reasonable spot on a donor's
own ranked preference list (Reform is Labour's rank-6 destination here — last, but still
above the `-1` "never" entries), some of that donor's transfer still reaches the leader,
because the leader is, definitionally, always tier 1 and therefore always an eligible
destination. Classic tactical voting is usually framed as *anti*-frontrunner coordination;
this implementation is better described as "vote consolidation toward whoever's viable, by
your own preference order" — which produces the right answer most of the time (leaders rarely
sit near the top of hostile donors' preference lists) but not by construction. Worth flagging
as a modelling choice rather than a bug — but a choice, not an inevitability.

## 5. Stage 4: turning percentages back into a map

The final step (`05_export_svg_output.py`) does two things. First, four specific seats —
**Great Yarmouth, Makerfield, Aberdeen South, Gorton and Denton** — get a bespoke, hand-built
flow matrix from `LocalFlows.xlsx` instead of the generic tribe-based projection, applied with
exactly the same row-stochastic mechanism as Section 2. Every other seat passes through
unchanged from the tactical-adjusted file. Four seats out of 632 needing individual,
judgement-based correction is a small number — but it's also exactly the kind of manual
patching that doesn't scale, and that Part 1's whole project (finding *why* a seat behaves
unusually, in data, rather than noticing it and hand-fixing it one at a time) is trying to
reduce the need for.

Second, projected **percentages** become projected **raw vote counts** by multiplying against
`prev_TOTAL` — that seat's total valid votes at the *previous* election:

In [14]:
vote_totals = pd.read_excel(PIPELINE / "data/raw/Tactical.xlsx", sheet_name="VoteTotals")
vote_totals[vote_totals["Seat"] == "Eastleigh"][["Seat", "prev_Electorate", "prev_TOTAL"]]

,Seat,prev_Electorate,prev_TOTAL
195,Eastleigh,70015,46420


There's no separate turnout model here at all — turnout is implicitly assumed identical to
last time, seat by seat. That's a defensible simplification (turnout is genuinely hard to
project, and errors here are usually smaller in seat-share terms than errors in who those
voters pick), but it is a real, load-bearing assumption: a seat with an unusually mobilised or
demobilised electorate next time around — plausible for exactly the kind of Muslim-vote and
Reform-curious seats Part 1 spent most of its time on — would have its vote counts (though not
its *shares*) systematically off by however much turnout actually moved.

## 6. Where this leaves things

None of Sections 2–5 needed to change to accommodate Part 1's revised groups — the pipeline's
actual machinery (row-stochastic flow matrices, gap-based winnability tiers, percentage→count
conversion) is agnostic to where the groups themselves came from. What's left to do, in
roughly the order it'd pay off:

1. **Re-derive the flow matrices** from the evidence-based group structure in Part 1, the same
   shape as the existing tribe sheets, estimated from real 2019→2024 transitions in each
   group's dominant seats rather than hand-set.
2. **Vectorize `03_monte_carlo.py`**, fixing the duplicated `"Restore"` entry from Section 3
   along the way, then wire it into `run_all.py` — Section 3 showed the ~40x speedup (roughly
   90 seconds down to ~2) that turns it from "too slow for a web request" into "fine for one."
   The statistics were never the obstacle; the loop was.
3. **Re-point the tactical voting parameters** at the current YouGov tracker wave directly,
   closing the Conservative→Reform gap identified in Section 4.
4. **Shrink the hand-tuned `LocalFlows.xlsx` seat list** by asking, for each of the four
   overridden seats, *what* about them the generic model gets wrong — the same
   evidence-driven approach Part 1 used to split Archetypes 1 and 3, rather than a one-off
   local patch.

That's the honest state of the prediction side of this project: a mechanically sound pipeline,
with one real bug, one calibration gap, one un-modelled assumption (static turnout), and one
genuine conceptual limitation (winnability isn't quite the same thing as tactical voting) —
all now specific and checkable rather than implicit.